In [13]:
import numpy as np
import time
import random

# ==========================================
# 1. EXPLICIT RADIX SORT (using NumPy)
# ==========================================
def explicit_radix_sort_16bit(arr):
    """
    An explicit 16-bit Radix Sort.
    Processes 8 bits per pass -> exactly 2 passes.
    Uses NumPy for vectorized bucketing to eliminate Python loop overhead.
    """
    # Create a mutable copy of the array
    output = np.empty_like(arr)
    
    # Pass 1: Sort by the lower 8 bits (bits 0-7)
    shift1 = 0
    bytes1 = (arr >> shift1) & 255
    # argsort on bytes1 gives us the stable bucket structure explicitly
    indices1 = np.argsort(bytes1, kind='stable')
    output = arr[indices1]
    
    # Pass 2: Sort by the upper 8 bits (bits 8-15)
    shift2 = 8
    bytes2 = (output >> shift2) & 255
    indices2 = np.argsort(bytes2, kind='stable')
    output = output[indices2]
    
    return output

# ==========================================
# 2. EXPLICIT TIMSORT
# ==========================================
MIN_MERGE = 32

def calc_min_run(n):
    """Calculates the minimum run length for Timsort."""
    r = 0
    while n >= MIN_MERGE:
        r |= n & 1
        n >>= 1
    return n + r

def insertion_sort(arr, left, right):
    """Standard insertion sort used by Timsort for small runs."""
    for i in range(left + 1, right + 1):
        j = i
        while j > left and arr[j] < arr[j - 1]:
            arr[j], arr[j - 1] = arr[j - 1], arr[j]
            j -= 1

def merge(arr, l, m, r):
    """Standard merge step used by Timsort to combine runs."""
    len1, len2 = m - l + 1, r - m
    left, right = arr[l:m+1], arr[m+1:r+1]
    
    i, j, k = 0, 0, l
    while i < len1 and j < len2:
        if left[i] <= right[j]:
            arr[k] = left[i]
            i += 1
        else:
            arr[k] = right[j]
            j += 1
        k += 1
        
    while i < len1:
        arr[k] = left[i]
        i += 1
        k += 1
        
    while j < len2:
        arr[k] = right[j]
        j += 1
        k += 1

def explicit_timsort(arr):
    """
    Explicit Timsort framework:
    1. Chop array into min-runs and sort them with Insertion Sort.
    2. Merge the sorted runs using Merge Sort logic.
    """
    n = len(arr)
    min_run = calc_min_run(n)
    
    # Step 1: Insertion sort individual small runs
    for start in range(0, n, min_run):
        end = min(start + min_run - 1, n - 1)
        insertion_sort(arr, start, end)
        
    # Step 2: Merge runs size by size (32 -> 64 -> 128...)
    size = min_run
    while size < n:
        for left in range(0, n, 2 * size):
            mid = min(n - 1, left + size - 1)
            right = min((left + 2 * size - 1), (n - 1))
            if mid < right:
                merge(arr, left, mid, right)
        size = 2 * size
    return arr

# ==========================================
# 3. BENCHMARK RUNNER
# ==========================================

# Using N=100,000 so the pure Python Timsort doesn't take too long
N = 100_000 
print(f"Generating {N:,} random 16-bit integers...\n")

# Base random data
raw_data = [random.randint(0, 65535) for _ in range(N)]
np_data = np.array(raw_data, dtype=np.uint16)
py_list = list(raw_data)

# 1. Run Explicit Radix Sort
start = time.perf_counter()
radix_res = explicit_radix_sort_16bit(np_data)
radix_time = time.perf_counter() - start
print(f"| Explicit Radix Sort (2-Pass NumPy) | Time: {radix_time:.5f} seconds")

# 2. Run Explicit Timsort 
start = time.perf_counter()
timsort_res = explicit_timsort(py_list)
timsort_time = time.perf_counter() - start
print(f"| Explicit Timsort    | Time: {timsort_time:.5f} seconds")


# 3. Run Python Built-in Timsort
start = time.perf_counter()
py_list.sort()  # In-place Timsort
builtin_timsort_time = time.perf_counter() - start
print(f"| Python Built-in Timsort    | Time: {builtin_timsort_time:.5f} seconds")


# Verification
print(f"\nAll three algorithms sorted identically: {list(radix_res) == timsort_res == py_list}")


Generating 100,000 random 16-bit integers...

| Explicit Radix Sort (2-Pass NumPy) | Time: 0.00451 seconds
| Explicit Timsort    | Time: 0.29946 seconds
| Python Built-in Timsort    | Time: 0.00143 seconds

All three algorithms sorted identically: True
